In [29]:
# Import para visualizar as pastas
import os
import sys

# Adiciona a pasta raiz do projeto ao path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from pathlib import Path

# Caminho onde serão salvo os resultados rsults/eda
results_path = Path("../results/eda")
results_path.mkdir(exist_ok=True)

In [ ]:
# import das bibliotecas utilizadas para exploração do dataframe
from src.load_data import load_raw_data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

/mnt/SegundoHD/Documentos/Projetos/ChatbotCredito/conda/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Carregando DataFrame base
df = load_raw_data()

Arquivo já existe, pulando donwload.
Arquivo salvo no projeto /mnt/SegundoHD/Documentos/Projetos/ChatbotCredito em: /mnt/SegundoHD/Documentos/Projetos/ChatbotCredito/data/raw


In [4]:
# Visualização do Dataframe
display(df.head(5))

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


Detalhes das features e target

---

| Variável                      | Descrição                             | Tipo                          |
| ----------------------------- | ------------------------------------- | ----------------------------- |
| person_age                    | Idade                                 | Numérico Discreto - Int       |
| person_income                 | Renda anual                           | Numérico Discreto - Int       |
| person_home_ownership         | Situação da moradia                   | Categórico Nominal - String   |
| person_emp_length             | Tempo de emprego (em anos)            | Numérico Contínuo - Float     |
| loan_intent                   | Intenção de empréstimo                | Categórico Nominal - String   |
| loan_grade                    | Grau do empréstimo                    | Categórico Ordinal - String   |
| loan_amnt                     | Valor do empréstimo                   | Numérico Discreto - Int       |
| loan_int_rate                 | Taxa de juro                          | Numérico Contínuo - Float     |
| loan_status                   | Situação do empréstimo                | Categórico Ordinal - Int      |
| loan_percent_income           | Renda percentual                      | Numérico Contínuo - Float     |
| cb_person_default_on_file     | Inadimplência histórica               | Categórico Ordinal - String   |
| cb_person_cred_hist_length    | Comprimento do histórico de crédito   | Numérico Discreto - Int       |

**Target - loan_status (0 significa não inadimplente, 1 significa inadimplente)**

In [5]:
# Verificação dos tipos de variáveis
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 3.0 MB


In [6]:
# Verificando se há dados ausentes
df.isna().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

Dados faltantes em person_emp_length e loan_int_rate

In [34]:
# Gerando mapa de calor (apenas colunas numéricas)
def geraMapaCalor(dataframe):
    plt.figure(figsize=(15,10))
    sns.heatmap(dataframe.corr(numeric_only=True), cmap=('Blues'), annot=True)
    plt.title('Correlação entre os atributos', size=20)
    #plt.show()

    # Salvando o mapa de calor na pasta results/eda
    file_name = results_path / 'mapa_calor_plot.png'
    plt.savefig(file_name)
    plt.close()

In [ ]:
geraMapaCalor(df)
# Mapa salvo em results/eda/mapa_calor_plot.png

In [37]:
# Pegando os melhores features para a classe
corr = df.corr(numeric_only=True)

target_corr = corr['loan_status'].sort_values(ascending=False)

# Salvando as info em results/eda
file_name = results_path / 'correlacoes_mapa_calor.csv'
target_corr.to_csv(file_name)

In [19]:
# Analise com variáveis categóricas
cat_cols = df.select_dtypes(include='str').columns

In [41]:
for col in cat_cols:
    result = pd.crosstab(df[col], df['loan_status'], normalize='index')

    file_name = results_path / f'{col}_crosstab.csv'
    result.to_csv(file_name)

In [42]:
n_cols = 2
n_rows = (len(cat_cols) + 1) // n_cols

fig, ax = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(12,5*n_rows))

ax = ax.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(x=col, hue='loan_status', data=df, ax=ax[i])
    ax[i].set_title(col)
    ax[i].tick_params(axis='x', rotation=45)

plt.tight_layout()


file_name = results_path / "grafico_crosstab.png"
plt.savefig(file_name)
plt.close()